# 04 — Classificazione Semantica nei Livelli Normativi

**VERSIONE CORRETTA** — Migrata da Gemini (quota esaurita) a **DeepSeek API** (~$3 per 3311 nodi).

## Fix applicati rispetto alla versione originale
1. Sostituita `google-generativeai` con `openai` SDK (DeepSeek è OpenAI-compatible)
2. Corretto rilevamento errori 429 nel loop principale
3. Aggiunto exponential backoff per rate limit
4. Rimosso delay fisso di 5s → delay adattivo (0.3s)
5. Corretta cella 3b: `sample_df` ora definita prima del loop

## Framework di classificazione

| Livello | Nome | Contenuto |
|---|---|---|
| **G1** | Principi fondamentali | Valori, obiettivi ultimi, diritti fondamentali |
| **G2** | Legge quadro | Regole generali, poteri, soglie, procedure principali |
| **G3** | Regolamenti tecnico-operativi | Dettagli tecnici, moduli, elenchi, tempistiche |
| **G4** | Linee guida e soft law | Raccomandazioni, comunicazioni, prassi |
| **G5** | Controllo e giurisprudenza | Sentenze, infrazione, sanzioni |

## 0. Setup

In [13]:
# DeepSeek usa l'SDK openai standard con base_url diverso — nessuna libreria proprietaria
# !pip install openai

import pandas as pd
import json
import os
import sys
import time
from openai import OpenAI, RateLimitError, APIStatusError

sys.path.append('..')
from config_golden_power import MATERIA_NAME

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path     = os.path.join('..', 'data', 'output', MATERIA_NAME)
preambles_file  = os.path.join(output_path, 'gephi_nodes_focal_preambles.csv')
titles_file     = os.path.join(output_path, 'gephi_nodes_focal_titled.csv')
output_file     = os.path.join(output_path, 'gephi_nodes_focal_classified.csv')
checkpoint_file = os.path.join(output_path, 'classification_checkpoint.csv')

# ── Configurazione API DeepSeek ───────────────────────────────────────────────
# 1. Registrati su https://platform.deepseek.com
# 2. Crea una API key su https://platform.deepseek.com/api_keys
# 3. Imposta la variabile d'ambiente: set DEEPSEEK_API_KEY=sk-...
#    oppure sostituisci os.environ.get(...) con la stringa diretta (solo per test locale)
client = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY', ''),
    base_url='https://api.deepseek.com',
)

MODEL            = 'deepseek-chat'  # DeepSeek V3 — ottimo per classificazione
MAX_TOKENS       = 256              # risposta JSON breve
DELAY_SECONDS    = 0.3              # DeepSeek ha rate limit generosi
CHECKPOINT_EVERY = 50

print(f"Input preamboli: {preambles_file}")
print(f"Input titoli:    {titles_file}")
print(f"Output:          {output_file}")
print(f"Modello:         {MODEL}")

Input preamboli: ..\data\output\golden_power\gephi_nodes_focal_preambles.csv
Input titoli:    ..\data\output\golden_power\gephi_nodes_focal_titled.csv
Output:          ..\data\output\golden_power\gephi_nodes_focal_classified.csv
Modello:         deepseek-chat


## 1. Caricamento e Preparazione Dati

In [14]:
preambles = pd.read_csv(preambles_file)
titles    = pd.read_csv(titles_file)[['Id', 'title']]

nodes = preambles.merge(titles, on='Id', how='left')

nodes['input_text']   = nodes['preamble'].fillna(nodes['title'])
nodes['input_source'] = nodes.apply(
    lambda r: 'preamble' if pd.notna(r['preamble'])
         else 'title'    if pd.notna(r['title'])
         else 'none',
    axis=1
)

print(f"Nodi totali: {len(nodes)}")
print()
print("Fonte del testo per classificazione:")
print(nodes['input_source'].value_counts().to_string())
print()
print(f"Nodi classificabili: {(nodes['input_source'] != 'none').sum()}")
print(f"Nodi non classificabili (nessun testo): {(nodes['input_source'] == 'none').sum()}")

Nodi totali: 4904

Fonte del testo per classificazione:
input_source
preamble    3177
none        1593
title        134

Nodi classificabili: 3311
Nodi non classificabili (nessun testo): 1593


## 2. Caricamento Checkpoint

In [15]:
if os.path.exists(checkpoint_file):
    checkpoint   = pd.read_csv(checkpoint_file)
    already_done = set(checkpoint['Id'])
    print(f"Checkpoint trovato: {len(already_done)} nodi già classificati")
    print(f"Nodi rimanenti:     {len(nodes) - len(already_done)}")
else:
    checkpoint   = pd.DataFrame(columns=[
        'Id', 'dominant_layer', 'purity',
        'G1', 'G2', 'G3', 'G4', 'G5',
        'motivation', 'input_source', 'classification_status'
    ])
    already_done = set()
    print("Nessun checkpoint trovato, si parte da zero")

nodes_todo = nodes[
    (~nodes['Id'].isin(already_done)) &
    (nodes['input_source'] != 'none')
].copy()

print(f"Da classificare ora: {len(nodes_todo)}")

Checkpoint trovato: 3311 nodi già classificati
Nodi rimanenti:     1593
Da classificare ora: 0


## 3. Prompt e Funzione di Classificazione

In [4]:
SYSTEM_PROMPT = """Sei un esperto di diritto dell'Unione Europea specializzato in tecnica legislativa e gerarchia delle fonti normative. Il tuo compito è classificare atti normativi UE secondo il framework Lamfalussy.

Rispondi ESCLUSIVAMENTE con un oggetto JSON valido, senza testo aggiuntivo, senza backtick, senza commenti."""

CLASSIFICATION_PROMPT = """Analizza il seguente atto normativo UE e distribuisci il suo contenuto sui 5 livelli del framework Lamfalussy.

LIVELLI:

G1 - PRINCIPI FONDAMENTALI
Contiene: valori fondanti, obiettivi ultimi dell'ordinamento, diritti e doveri fondamentali. Il livello più stabile e difficile da modificare.
Tipicamente: Trattati, Carta dei diritti fondamentali, principi generali del diritto UE.
Segnali: "is guaranteed", "constitutes a fundamental objective", "is an exclusive competence", "shall be prohibited", "Having regard to the Treaty"

G2 - LEGGE QUADRO
Contiene: traduzione dei principi in regole generali operative. Definisce poteri, soglie, procedure principali, istituzione di organi. Livello del dibattito parlamentare (PE + Consiglio).
Tipicamente: Regolamenti del PE e del Consiglio, Direttive.
Segnali: "the Commission shall", "Member States shall", "is hereby established", "the threshold shall be", "ordinary legislative procedure"

G3 - REGOLAMENTI TECNICO-OPERATIVI
Contiene: dettagli tecnici per rendere operativa la legge quadro. Moduli, elenchi, codici, tempistiche, standard tecnici. Livello delle agenzie e autorità indipendenti.
Tipicamente: Regolamenti delegati, Regolamenti di esecuzione, Decisioni tecniche della Commissione.
Segnali: "pursuant to Article X of Regulation", "the standard form", "the list set out in the Annex", "within X days", "Commission Delegated", "Commission Implementing"

G4 - LINEE GUIDA E SOFT LAW
Contiene: interpretazioni delle regole, prassi applicativa, raccomandazioni non vincolanti.
Tipicamente: Raccomandazioni, Comunicazioni della Commissione, Linee guida di autorità di vigilanza.
Segnali: "recommends", "should be understood as", "for the purposes of applying", "non-binding", "guidelines", "best practice"

G5 - CONTROLLO E GIURISPRUDENZA
Contiene: applicazione forzosa delle regole, sanzioni, interpretazione autentica in caso di conflitto.
Tipicamente: Sentenze CGUE, Decisioni di infrazione, Provvedimenti sanzionatori.
Segnali: "the Court rules", "the action is dismissed", "infringement", "the fine", "judgment", "annuls"

ISTRUZIONI:
- Assegna una percentuale a ciascun livello (G1, G2, G3, G4, G5).
- Le percentuali devono sommare esattamente a 100.
- Usa l'intera scala: distribuzioni come 70/20/10 o 60/30/10 sono più informative di 90/10/0.
- Riserva 100% o 90% SOLO se il testo è inequivocabilmente mono-livello (es. sentenza pura, trattato puro).
- Un Regolamento PE+Consiglio che cita i Trattati e contiene anche allegati tecnici potrebbe essere 65% G2, 15% G1, 20% G3.
- Un atto che mescola principi e regole operative avrà percentuali distribuite su più livelli — questo è un risultato interpretabile, non un errore.
- La motivazione deve essere in italiano, massimo 8 parole.

TESTO DELL'ATTO:
{testo}

Rispondi SOLO con questo JSON (percentuali intere che sommano a 100):
{{"G1": 0, "G2": 0, "G3": 0, "G4": 0, "G5": 0, "motivation": "..."}}"""


def classify_act(text, source='preamble', max_retries=3):
    """Classifica un atto normativo usando DeepSeek API con retry automatico.

    DeepSeek è compatibile con il formato OpenAI: stesse eccezioni, stesso schema messaggi.
    Returns: (dist, dominant_layer, purity, motivation, status)
    """
    if pd.isna(text) or text == '':
        return None, None, None, None, 'no_text'

    text_truncated = str(text)[:1500]
    prompt = CLASSIFICATION_PROMPT.format(testo=text_truncated)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                temperature=0.1,
                response_format={'type': 'json_object'},  # JSON mode nativo DeepSeek
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': prompt},
                ]
            )

            content = response.choices[0].message.content.strip()

            # Pulizia difensiva nel caso arrivino backtick residui
            content = content.replace('```json', '').replace('```', '').strip()

            parsed = json.loads(content)

            layers = ['G1', 'G2', 'G3', 'G4', 'G5']
            dist   = {}
            for l in layers:
                val     = parsed.get(l, 0)
                dist[l] = float(val) if val is not None else 0.0

            total = sum(dist.values())
            if total == 0:
                return None, None, None, None, 'invalid_distribution'

            # Normalizza se non somma esattamente a 100
            if abs(total - 100) > 1:
                dist = {l: dist[l] / total * 100 for l in layers}

            dominant_layer = max(dist, key=dist.get)
            purity         = dist[dominant_layer] / 100.0
            motivation     = parsed.get('motivation', '')

            return dist, dominant_layer, purity, motivation, 'ok'

        except json.JSONDecodeError:
            if attempt == max_retries - 1:
                return None, None, None, None, 'json_error'
            time.sleep(1)

        except RateLimitError:
            # 429 — backoff esponenziale: 30s, 60s, 120s
            wait = 30 * (2 ** attempt)
            print(f"  [RATE LIMIT] Attendo {wait}s (tentativo {attempt+1}/{max_retries})...")
            time.sleep(wait)
            if attempt == max_retries - 1:
                return None, None, None, None, 'rate_limit'

        except APIStatusError as e:
            return None, None, None, None, f'api_error_{e.status_code}'

        except Exception as e:
            return None, None, None, None, f'error: {str(e)[:80]}'


# ── Test su atto seed noto ────────────────────────────────────────────────────
if 'Label' in nodes.columns:
    test_row = nodes[nodes['Label'] == '32019R0452']
else:
    test_row = nodes.head(1)

if len(test_row) > 0:
    sample = test_row['input_text'].iloc[0]
    celex  = test_row['Label'].iloc[0] if 'Label' in test_row.columns else 'primo nodo'
    print(f"Test su {celex} (FDI Screening)...")
    dist, dominant, purity, motiv, status = classify_act(sample)
    print(f"  Status: {status}")
    if status == 'ok':
        print(f"  Dominante:     {dominant}  (purezza={purity:.2f})")
        print(f"  Distribuzione: {dist}")
        print(f"  Motiv:         {motiv}")
    else:
        print(f"  ERRORE — controlla DEEPSEEK_API_KEY e saldo account su platform.deepseek.com")

Test su 32019R0452 (FDI Screening)...
  Status: ok
  Dominante:     G2  (purezza=0.70)
  Distribuzione: {'G1': 20.0, 'G2': 70.0, 'G3': 5.0, 'G4': 5.0, 'G5': 0.0}
  Motiv:         Legge quadro con riferimenti ai trattati


## 3b. Test su Campione Stratificato

Prima di classificare tutti i nodi, verifica la qualità delle classificazioni su un campione di 20 nodi.
**Esegui questa cella e controlla manualmente le classificazioni prima di procedere con la cella 4.**

In [8]:
SAMPLE_PER_TYPE = 2

# Campione stratificato per tipo di atto
sample_parts = []

if 'resource_legal_type' in nodes.columns:
    for t in nodes['resource_legal_type'].dropna().unique():
        sub = nodes[
            (nodes['resource_legal_type'] == t) &
            (nodes['input_source'] != 'none')
        ]
        if len(sub) >= SAMPLE_PER_TYPE:
            sample_parts.append(sub.sample(SAMPLE_PER_TYPE, random_state=42))
        elif len(sub) > 0:
            sample_parts.append(sub)
    sample_df = pd.concat(sample_parts).drop_duplicates(subset=['Id']).head(20)
else:
    sample_df = nodes[nodes['input_source'] != 'none'].sample(
        min(20, len(nodes)), random_state=42
    )

print(f"Campione di test: {len(sample_df)} nodi")
if 'resource_legal_type' in sample_df.columns:
    print(f"Per tipo: {dict(sample_df['resource_legal_type'].value_counts())}")
print(f"Per fonte: {dict(sample_df['input_source'].value_counts())}")
print()

for i, (_, row) in enumerate(sample_df.iterrows()):
    label = row.get('Label', row['Id'])
    dist, dominant, purity, motiv, status = classify_act(row['input_text'], row['input_source'])
    purity_str = f"{purity:.2f}" if purity is not None else 'N/A'
    print(f"[{i+1:>2}/{len(sample_df)}] {str(label):<25} → {dominant} (purezza={purity_str}) [{status}]")
    if status == 'ok':
        print(f"        dist={dist}")
        print(f"        motiv={motiv}")
    time.sleep(DELAY_SECONDS)

Campione di test: 20 nodi
Per fonte: {'preamble': np.int64(19), 'title': np.int64(1)}

[ 1/20] 32019L2235                → G2 (purezza=0.70) [ok]
        dist={'G1': 20.0, 'G2': 70.0, 'G3': 10.0, 'G4': 0.0, 'G5': 0.0}
        motiv=Legge quadro con principi e dettagli tecnici
[ 2/20] 32023L2413                → G2 (purezza=0.70) [ok]
        dist={'G1': 20.0, 'G2': 70.0, 'G3': 5.0, 'G4': 5.0, 'G5': 0.0}
        motiv=Legge quadro con principi fondanti e dettagli tecnici
[ 3/20] 32023D0921                → G2 (purezza=0.60) [ok]
        dist={'G1': 20.0, 'G2': 60.0, 'G3': 10.0, 'G4': 10.0, 'G5': 0.0}
        motiv=Atto legislativo con principi e regole operative
[ 4/20] 32025R0849                → G3 (purezza=0.80) [ok]
        dist={'G1': 10.0, 'G2': 0.0, 'G3': 80.0, 'G4': 10.0, 'G5': 0.0}
        motiv=Atto tecnico con riferimenti a principi e linee guida
[ 5/20] 32015D1065                → G2 (purezza=0.80) [ok]
        dist={'G1': 20.0, 'G2': 80.0, 'G3': 0.0, 'G4': 0.0, 'G5': 0.0}
 

## 4. Classificazione Completa con Checkpoint

Con 3311 nodi e delay di 0.3s il tempo stimato è circa **17 minuti**.

Se viene interrotto, riesegui questa cella: ripartirà dal checkpoint automaticamente.

In [8]:
results = []
total   = len(nodes_todo)
n_ok    = 0
n_err   = 0

print(f"Inizio classificazione: {total} nodi")
print(f"Tempo stimato: ~{total * DELAY_SECONDS / 60:.0f} minuti\n")

for i, (_, row) in enumerate(nodes_todo.iterrows()):
    node_id = row['Id']
    text    = row['input_text']
    source  = row['input_source']

    dist, dominant, purity, motivation, status = classify_act(text, source)

    if status == 'ok':
        n_ok += 1
    else:
        n_err += 1
        if status == 'rate_limit':
            print(f"  [RATE LIMIT esaurito] Pausa lunga 120s...")
            time.sleep(120)
        elif status.startswith('api_error'):
            print(f"  [{status}] Pausa 30s...")
            time.sleep(30)

    results.append({
        'Id':                    node_id,
        'dominant_layer':        dominant,
        'purity':                purity,
        'G1': dist['G1'] if dist else None,
        'G2': dist['G2'] if dist else None,
        'G3': dist['G3'] if dist else None,
        'G4': dist['G4'] if dist else None,
        'G5': dist['G5'] if dist else None,
        'motivation':            motivation,
        'input_source':          source,
        'classification_status': status,
    })

    if (i + 1) % 10 == 0 or (i + 1) == total:
        pct = (i + 1) / total * 100
        print(f"  [{i+1:>4}/{total}] {pct:5.1f}%  ok: {n_ok}  errori: {n_err}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        batch              = pd.DataFrame(results)
        checkpoint = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
        checkpoint.to_csv(checkpoint_file, index=False)
        results = []
        print(f"  --> Checkpoint salvato ({len(checkpoint)} nodi totali)")

    time.sleep(DELAY_SECONDS)

# Salva risultati finali
if results:
    batch            = pd.DataFrame(results)
    checkpoint_final = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
else:
    checkpoint_final = pd.read_csv(checkpoint_file)

checkpoint_final.to_csv(checkpoint_file, index=False)
print(f"\nClassificazione completata.")
print(f"  OK:     {(checkpoint_final['classification_status'] == 'ok').sum()}")
print(f"  Errori: {(checkpoint_final['classification_status'] != 'ok').sum()}")
print()
print("Distribuzione layer dominante:")
print(checkpoint_final['dominant_layer'].value_counts().to_string())

Inizio classificazione: 3111 nodi
Tempo stimato: ~16 minuti

  [  10/3111]   0.3%  ok: 10  errori: 0
  [  20/3111]   0.6%  ok: 20  errori: 0
  [  30/3111]   1.0%  ok: 30  errori: 0
  [  40/3111]   1.3%  ok: 40  errori: 0
  [  50/3111]   1.6%  ok: 50  errori: 0
  --> Checkpoint salvato (250 nodi totali)
  [  60/3111]   1.9%  ok: 60  errori: 0
  [  70/3111]   2.3%  ok: 70  errori: 0
  [  80/3111]   2.6%  ok: 80  errori: 0
  [  90/3111]   2.9%  ok: 90  errori: 0
  [ 100/3111]   3.2%  ok: 100  errori: 0
  --> Checkpoint salvato (300 nodi totali)
  [ 110/3111]   3.5%  ok: 110  errori: 0
  [ 120/3111]   3.9%  ok: 120  errori: 0
  [ 130/3111]   4.2%  ok: 130  errori: 0
  [ 140/3111]   4.5%  ok: 140  errori: 0
  [ 150/3111]   4.8%  ok: 150  errori: 0
  --> Checkpoint salvato (350 nodi totali)
  [ 160/3111]   5.1%  ok: 160  errori: 0
  [ 170/3111]   5.5%  ok: 170  errori: 0
  [ 180/3111]   5.8%  ok: 180  errori: 0
  [ 190/3111]   6.1%  ok: 190  errori: 0
  [ 200/3111]   6.4%  ok: 200  errori: 0

In [16]:
# Riclassifica i nodi falliti nel checkpoint
checkpoint = pd.read_csv(checkpoint_file)

failed = checkpoint[checkpoint['classification_status'] != 'ok'].copy()
print(f"Nodi da riclassificare: {len(failed)}")

# Recupera il testo originale da nodes
failed_with_text = failed[['Id', 'input_source']].merge(
    nodes[['Id', 'input_text']], on='Id', how='left'
)

results_retry = []
n_ok = 0

for i, (_, row) in enumerate(failed_with_text.iterrows()):
    dist, dominant, purity, motivation, status = classify_act(
        row['input_text'], row['input_source']
    )
    if status == 'ok':
        n_ok += 1

    results_retry.append({
        'Id':                    row['Id'],
        'dominant_layer':        dominant,
        'purity':                purity,
        'G1': dist['G1'] if dist else None,
        'G2': dist['G2'] if dist else None,
        'G3': dist['G3'] if dist else None,
        'G4': dist['G4'] if dist else None,
        'G5': dist['G5'] if dist else None,
        'motivation':            motivation,
        'input_source':          row['input_source'],
        'classification_status': status,
    })

    if (i + 1) % 10 == 0:
        print(f"  [{i+1:>3}/{len(failed)}]  ok: {n_ok}")

    time.sleep(DELAY_SECONDS)

# Sostituisce le righe fallite nel checkpoint
retry_df = pd.DataFrame(results_retry)
checkpoint = checkpoint[checkpoint['classification_status'] == 'ok']  
checkpoint = pd.concat([checkpoint, retry_df]).drop_duplicates(subset=['Id'])
checkpoint.to_csv(checkpoint_file, index=False)

print(f"\nRetry completato.")
print(f"  OK ora:   {(checkpoint['classification_status'] == 'ok').sum()}")
print(f"  Ancora errori: {(checkpoint['classification_status'] != 'ok').sum()}")

Nodi da riclassificare: 0

Retry completato.
  OK ora:   3311
  Ancora errori: 0


## 5. Export e Calcolo Metriche

In [ ]:
class_df = pd.read_csv(checkpoint_file)[[
    'Id', 'dominant_layer', 'purity',
    'G1', 'G2', 'G3', 'G4', 'G5',
    'motivation', 'input_source', 'classification_status'
]]

class_df = class_df.rename(columns={
    'dominant_layer': 'layer',
    'purity':         'layer_confidence',
})

class_df = class_df.drop(columns=['input_source'], errors='ignore')
nodes_classified = nodes.merge(class_df, on='Id', how='left')

nodes_classified = nodes.merge(class_df, on='Id', how='left')

# Tronca il titolo per Gephi: salta il prefisso e prendi le prime 8 parole significative
def truncate_title(t, skip=4, keep=8):
    if pd.isna(t):
        return None
    words = str(t).split()
    meaningful = words[skip:skip+keep]
    return ' '.join(meaningful) + '...' if meaningful else str(t)[:50]

# Rimuovi colonne non rilevanti
nodes_classified = nodes_classified.drop(columns=[
    'input_text',
    'preamble_status',
    'preamble_length', 
    'input_source',
    'classification_status',
], errors='ignore')

# Rimuovi nodi senza informazioni
nodes_classified = nodes_classified[nodes_classified['layer'].notna()]

nodes_classified.to_csv(output_file, index=False)

print(f"File salvato: {output_file}")
print(f"  Classificati: {nodes_classified['layer'].notna().sum()}")
print(f"\nDistribuzione layer dominante:")
print(nodes_classified['layer'].value_counts().to_string())
print(f"\nPurezza media per layer:")
print(nodes_classified.groupby('layer')['layer_confidence'].mean().round(2).sort_index().to_string())

File salvato: ..\data\output\golden_power\gephi_nodes_focal_classified.csv
  Classificati: 3311

Distribuzione layer dominante:
layer
G2    1822
G3    1228
G5     218
G4      37
G1       6

Purezza media per layer:
layer
G1    0.65
G2    0.69
G3    0.67
G4    0.73
G5    0.83


In [26]:
edges = pd.read_csv(os.path.join(output_path, 'gephi_edges_focal.csv'))

# Tieni solo gli edge dove sia source che target sono nel grafo filtrato
valid_ids = set(nodes_classified['Id'])
edges_filtered = edges[
    edges['Source'].isin(valid_ids) & 
    edges['Target'].isin(valid_ids)
]

print(f"Edge totali:   {len(edges)}")
print(f"Edge filtrati: {len(edges_filtered)}")
print(f"Edge rimossi:  {len(edges) - len(edges_filtered)}")

edges_filtered.to_csv(
    os.path.join(output_path, 'gephi_edges_focal_classified.csv'), 
    index=False
)

Edge totali:   25743
Edge filtrati: 13781
Edge rimossi:  11962


In [27]:
import numpy as np

classified = nodes_classified[nodes_classified['layer'].notna()].copy()
LAYERS = ['G1', 'G2', 'G3', 'G4', 'G5']

def shannon_entropy(row):
    probs = np.array([row[l] for l in LAYERS], dtype=float) / 100.0
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs)) if len(probs) > 0 else 0.0

classified['layer_entropy'] = classified.apply(shannon_entropy, axis=1)

PURITY_THRESHOLD = 0.7
hybrid = classified[classified['layer_confidence'] < PURITY_THRESHOLD]

print("=== ANALISI LAMFALUSSY ===")
print()
print(f"Soglia purezza: {PURITY_THRESHOLD}")
print(f"Atti puri  (purezza >= {PURITY_THRESHOLD}): {len(classified) - len(hybrid)} ({(len(classified)-len(hybrid))/len(classified)*100:.1f}%)")
print(f"Atti ibridi (purezza < {PURITY_THRESHOLD}):  {len(hybrid)} ({len(hybrid)/len(classified)*100:.1f}%)")
print()
print("Purezza media per layer:")
print(classified.groupby('layer')['layer_confidence'].mean().round(2).sort_index().to_string())
print()
print("Entropia media per layer (0=atto puro, 2.32=uniforme):")
print(classified.groupby('layer')['layer_entropy'].mean().round(2).sort_index().to_string())
print()
print("Co-occurrence rate G2-G3 (mescolanza più critica):")
cooc_g2g3 = ((classified['G2'] > 20) & (classified['G3'] > 20)).sum() / len(classified) * 100
print(f"  Atti con G2>20% E G3>20%: {cooc_g2g3:.1f}%")

nodes_classified = nodes_classified.copy()
nodes_classified.loc[classified.index, 'layer_entropy'] = classified['layer_entropy']
nodes_classified.to_csv(output_file, index=False)
print(f"\nFile aggiornato con colonna layer_entropy: {output_file}")

=== ANALISI LAMFALUSSY ===

Soglia purezza: 0.7
Atti puri  (purezza >= 0.7): 2178 (65.8%)
Atti ibridi (purezza < 0.7):  1133 (34.2%)

Purezza media per layer:
layer
G1    0.65
G2    0.69
G3    0.67
G4    0.73
G5    0.83

Entropia media per layer (0=atto puro, 2.32=uniforme):
layer
G1    1.18
G2    1.27
G3    1.29
G4    0.94
G5    0.64

Co-occurrence rate G2-G3 (mescolanza più critica):
  Atti con G2>20% E G3>20%: 4.3%

File aggiornato con colonna layer_entropy: ..\data\output\golden_power\gephi_nodes_focal_classified.csv


## 6. Analisi Ambiguità

In [24]:
AMBIGUITY_THRESHOLD = 0.6

classified = nodes_classified[nodes_classified['layer'].notna()].copy()
ambiguous  = classified[classified['layer_confidence'] < AMBIGUITY_THRESHOLD]

print(f"=== ANALISI AMBIGUITÀ (soglia confidenza < {AMBIGUITY_THRESHOLD}) ===")
print()
print(f"Atti ambigui:      {len(ambiguous)} ({len(ambiguous)/len(classified)*100:.1f}%)")
print(f"Atti non ambigui:  {len(classified) - len(ambiguous)} ({(len(classified)-len(ambiguous))/len(classified)*100:.1f}%)")
print()
print("Atti ambigui per layer assegnato:")
print(ambiguous['layer'].value_counts().to_string())
print()
print("Confidenza media per layer:")
conf_stats = classified.groupby('layer')['layer_confidence'].agg(['mean', 'median', 'min'])
print(conf_stats.round(2).to_string())
print()
print("Esempi di atti ambigui (confidenza più bassa):")
label_col = 'Label' if 'Label' in ambiguous.columns else 'Id'
esempi = ambiguous.nsmallest(10, 'layer_confidence')[[label_col, 'layer', 'layer_confidence', 'motivation']]
for _, r in esempi.iterrows():
    print(f"  {r['layer']} | conf={r['layer_confidence']:.2f} | {r[label_col]} | {str(r['motivation'])[:80]}")

=== ANALISI AMBIGUITÀ (soglia confidenza < 0.6) ===

Atti ambigui:      214 (6.5%)
Atti non ambigui:  3097 (93.5%)

Atti ambigui per layer assegnato:
layer
G3    147
G2     33
G5     24
G4      9
G1      1

Confidenza media per layer:
       mean  median  min
layer                   
G1     0.65     0.7  0.4
G2     0.69     0.7  0.3
G3     0.67     0.6  0.3
G4     0.73     0.7  0.4
G5     0.83     0.9  0.4

Esempi di atti ambigui (confidenza più bassa):
  G3 | conf=0.30 | 32022D1920 | Decisione di infrazione con elementi procedurali e tecnici
  G3 | conf=0.30 | 32007D0258 | Decisione di infrazione con elementi procedurali e tecnici
  G3 | conf=0.30 | 32018D0860 | Decisione di infrazione con elementi procedurali e tecnici
  G3 | conf=0.30 | 32014D0884 | Decisione di infrazione con riferimenti procedurali e tecnici
  G3 | conf=0.30 | 32010D0395 | Decisione di infrazione con elementi procedurali e tecnici
  G3 | conf=0.30 | 32022D0763 | Decisione su aiuti di Stato con riferimenti procedur